### 1. conda 환경 생성 및 활성화
터미널에서 실행:  
conda create -n autogluon_env python=3.10 -y

conda activate autogluon_env

이후 환경 안에서 AutoGluon 설치:

pip install -U pip wheel setuptools

pip install autogluon.tabular -q 


In [ ]:
# 파일: autogluon_run.py (예시)

import pandas as pd
from autogluon.tabular import TabularPredictor

# 1) 데이터 로드
train_path = "../data/train.tsv"
test_path = "../data/test.tsv"

# TSV 이므로 sep="\t" 사용
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# 2) 타깃 / ID 컬럼 이름 설정
TARGET_COL = "price"   # 실제 타깃 컬럼명으로 바꿔 주세요
ID_COL = None           # 제출용 ID 컬럼명으로 바꿔 주세요 (없으면 None)

# 3) AutoGluon 학습
# presets / time_limit 등은 상황에 맞게 조절
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type=None,          # 회귀/분류 자동 추론, 명시하고 싶으면 "regression"/"multiclass"/"binary"
    path="autogluon_models"     # 모델이 저장될 폴더
).fit(
    train_data=train_df,
    presets="medium_quality_faster_train",  # 빠른 실험용
    time_limit=3600,                        # 최대 1시간 (초 단위), 필요에 따라 조정
)

# 4) 리더보드 확인 (optional)
leaderboard_df = predictor.leaderboard(silent=True)
print(leaderboard_df.head())

# 5) 테스트 데이터 예측
test_preds = predictor.predict(test_df)

# 6) 제출 파일 생성
if ID_COL in test_df.columns:
    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_preds
    })
else:
    # ID 컬럼이 없다면 단순히 index 기반으로 생성
    submission = pd.DataFrame({
        "id": range(len(test_preds)),
        TARGET_COL: test_preds
    })

submission_path = "submission_autogluon.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       23.29 GB / 31.91 GB (73.0%)
Disk Space Avail:   342.83 GB / 465.09 GB (73.7%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'


Train shape: (1482535, 8)
Test shape: (693359, 7)


Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\big20\git\big20-ML-project2-team3\MercariPriceSuggestion\src\autogluon_models"
Train Data Rows:    1482535
Train Data Columns: 7
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and label-values can't be converted to int).
	Label info (max, min, mean, stddev): (2009.0, 0.0, 26.73752, 38.58607)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    24383.29 MB
	Train Data (Original)  Memory Usage: 667.54 MB (2.7% of available memory)
	Inferring data type of each feature based on column va